# Détourage des nageoires par détection d'objet saillant (U-2-Net)

Adapté du notebook Kaggle [Remove background - Salient Object Detection](https://www.kaggle.com/code/remekkinas/remove-background-salient-object-detection) de remekkinas (licence Apache 2.0).

## 1. SETUP NOTEBOOK (MODULES)
Let's clone U-2-Net repository

In [ ]:
import os
import sys
from pathlib import Path

# Code et poids U-2-Net hors du dépôt :
# git clone https://github.com/xuebinqin/U-2-Net ~/models/U-2-Net
U2NET_DIR = Path(os.path.expanduser("~/models/U-2-Net"))
sys.path.append(str(U2NET_DIR))

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
DATA = ROOT / "data"

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from data_loader import RescaleT, SalObjDataset, ToTensorLab
from model import U2NET, U2NETP
from PIL import Image as Img
from torch.utils.data import DataLoader
from torchvision import transforms

from dorsalmatch.embedder import pick_device

device = pick_device()
device

## 2. TAKE CANDIDATES FOR BACKGROUND REMOVAL

Photos du catalogue Miraceti (`data/curated/catalogue`), récupérées avec `python -m pipeline.storage pull`.

In [ ]:
LIVRAISON = "2026-11-06_catalogue_tt"
train_df = pd.read_csv(DATA / "curated/catalogue/manifests" / f"{LIVRAISON}.csv")

input_path = DATA / "curated/catalogue/photos" / LIVRAISON

In [ ]:
train_df.head()

On prend un des individus les plus photographiés.

In [ ]:
train_df.individu.value_counts().head(10)

Un seul individu pour juger la qualité du masque.

In [ ]:
img_to_draw = []
for individu in train_df.individu.value_counts().index[:3]:
    photos = train_df[train_df.individu == individu]
    img_to_draw.extend(
        [
            str(input_path / str(row.individu) / str(row.fichier_origine))
            for row in photos.sample(min(25, len(photos)), random_state=0).itertuples()
        ]
    )
len(img_to_draw)

In [ ]:
img_to_draw

In [ ]:
img_to_draw[0].split("/")[-1].split("_")[:2]

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(20, 20), squeeze=False)

for idx, img in enumerate(img_to_draw):
    cote, ind_code = img.split("/")[-1].split("_")[:2]

    i = idx // 3
    j = idx % 3
    image = Img.open(img)
    iar_shp = np.array(image).shape
    axes[i, j].axis("off")
    axes[i, j].imshow(image)
    axes[i, j].set_title(f"individu: {ind_code} - côté: {cote}")

for ax in axes.flat:
    ax.axis("off")

plt.subplots_adjust(wspace=0.01, hspace=0.01)
plt.show()

## 3. SOD - U2-Net PREDICTION

In [ ]:
THRESHOLD = 0.3

In [ ]:
def normPRED(d):
    ma = torch.max(d)
    mi = torch.min(d)
    return (d - mi) / (ma - mi)


def pred_unet(model, imgs):

    test_salobj_dataset = SalObjDataset(
        img_name_list=imgs,
        lbl_name_list=[],
        transform=transforms.Compose([RescaleT(320), ToTensorLab(flag=0)]),
    )
    test_salobj_dataloader = DataLoader(
        test_salobj_dataset, batch_size=1, shuffle=False, num_workers=0
    )

    for _i_test, data_test in enumerate(test_salobj_dataloader):
        inputs_test = data_test["image"]
        inputs_test = inputs_test.type(torch.FloatTensor)

        inputs_test = inputs_test.to(device)

        with torch.no_grad():
            d1, d2, d3, d4, d5, d6, d7 = model(inputs_test)

        predict = d5[:, 0, :, :]
        predict = normPRED(predict)

        del d1, d2, d3, d4, d5, d6, d7

        predict = predict.squeeze()
        predict_np = predict.cpu().data.numpy()

        # Masked image - using threshold you can soften/sharpen mask boundaries
        predict_np[predict_np > THRESHOLD] = 1
        predict_np[predict_np <= THRESHOLD] = 0
        mask = Img.fromarray(predict_np * 255).convert("RGB")
        image = Img.open(imgs[0])
        imask = mask.resize((image.width, image.height), resample=Img.BILINEAR)
        back = Img.new("RGB", (image.width, image.height), (255, 255, 255))
        mask = imask.convert("L")
        im_out = Img.composite(image, back, mask)

        # Salient mask
        salient_mask = np.array(image)
        mask_layer = np.array(imask)
        mask_layer[mask_layer == 255] = 50  # offest on RED channel
        salient_mask[:, :, 0] += mask_layer[:, :, 0]
        salient_mask = np.clip(salient_mask, 0, 255)

    return np.array(im_out), np.array(image), np.array(salient_mask), np.array(mask)

In [ ]:
UNET2_SMALL = False

In [ ]:
%%capture

if UNET2_SMALL:
    model_dir = (
        U2NET_DIR / "saved_models/u2netp/u2netp.pth"
    )  # Faster ... a lot (!) but less accurate
    net = U2NETP(3, 1)
else:
    model_dir = U2NET_DIR / "saved_models/u2net/u2net.pth"
    net = U2NET(3, 1)

net.load_state_dict(torch.load(model_dir, map_location=device))
net.to(device)
net.eval()

## 4. U2-Net RESULT VISUALIZATION

In [ ]:
fig, axes = plt.subplots(
    len(img_to_draw), 1, figsize=(15, 5 * len(img_to_draw)), squeeze=False
)

for idx, img in enumerate(img_to_draw):
    image, im_oryg, sal_map, mask = pred_unet(net, [img_to_draw[idx]])
    result = np.concatenate((im_oryg, sal_map, image), axis=1)
    result_img = Img.fromarray(result)
    axes[idx, 0].axis("off")
    axes[idx, 0].imshow(result_img)

    # on ajoute les infos permettant de repérer l'individu
    cote, ind_code = img.split("/")[-1].split("_")[:2]
    axes[idx, 0].set_title(f"individu: {ind_code} - côté: {cote}")


plt.subplots_adjust(wspace=0.05, hspace=0.05)
plt.show()

## 5. IDEAS FOR IMPROVEMENT

### A. REMOVE WHITE AREA OUTSIDE SAILENT MAP - PROTOTYPE
Let's find bbox for Sailent Map and remove white area outside it. This is just prototype - some code refactoring is needed. 

In [ ]:
fig, axes = plt.subplots(len(img_to_draw), 3, figsize=(30, int(6 * len(img_to_draw))))

for idx, img in enumerate(img_to_draw):
    # on ajoute les infos permettant de repérer l'individu
    cote, ind_code = img.split("/")[-1].split("_")[:2]

    image, im_oryg, sal_map, mask = pred_unet(net, [img_to_draw[idx]])

    # ymin = np.nonzero((mask[:] != 0).argmax(axis = 1))[0][0]
    # ymax = np.nonzero((mask[:] != 0).argmax(axis = 1))[0][-1]
    # xmin = np.nonzero((mask[:] != 0).argmax(axis = 0))[0][0]
    # xmax = np.nonzero((mask[:] != 0).argmax(axis = 0))[0][-1]

    ymin = np.min(np.nonzero(mask[:] == 255)[0])
    ymax = np.max(np.nonzero(mask[:] == 255)[0])
    xmin = np.min(np.nonzero(mask[:] == 255)[1])
    xmax = np.max(np.nonzero(mask[:] == 255)[1])

    img = cv2.rectangle(image.copy(), (xmin, ymin), (xmax, ymax), (255, 0, 0), 2)

    crop_img = image[ymin:ymax, xmin:xmax]
    crop_img = cv2.resize(
        crop_img, (image.shape[1], image.shape[0]), interpolation=cv2.INTER_AREA
    )

    result = np.concatenate((img, crop_img), axis=1)
    result_img = Img.fromarray(result)

    axes[idx, 0].imshow(Img.fromarray(sal_map))
    axes[idx, 1].imshow(Img.fromarray(img))
    axes[idx, 2].imshow(Img.fromarray(crop_img))
    axes[idx, 0].set_title(f"individu: {ind_code} - côté: {cote}")

plt.subplots_adjust(wspace=0.1, hspace=0.1)
plt.show()